# 💰 Financial Health Prediction — Advanced Optimized Pipeline

This notebook implements all best-practice strategies to maximize the **Weighted F1 Score**:

| Step | Strategy |
|------|----------|
| 1 | Smart preprocessing + missing value handling |
| 2 | Advanced feature engineering (interactions, log transforms) |
| 3 | Class weight balancing |
| 4 | XGBoost + LightGBM + CatBoost with StratifiedKFold |
| 5 | Optuna hyperparameter tuning |
| 6 | Stacking meta-model on OOF predictions |
| 7 | Pseudo-labeling with high-confidence test samples |
| 8 | Final submission generation |

## 1. 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
N_FOLDS = 5

print("✅ Libraries loaded.")

## 2. 📂 Load Data

In [ ]:
train_df = pd.read_csv('Train.csv')
test_df  = pd.read_csv('Test.csv')
ss       = pd.read_csv('Sample_Submission.csv')

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
print(f"\nTarget distribution:")
print(train_df['Target'].value_counts())
print(f"\nClass imbalance ratio:")
print(train_df['Target'].value_counts(normalize=True).round(3))

## 3. 🔧 Preprocessing

We combine train + test before encoding to ensure consistent category mapping.

**Encoding strategy:**
- **Ownership columns** (`Have now / Used to have / Never had`) → ordinal 0/1/2
- **Yes/No columns** → binary 0/1, unknown → -1
- **Country** → one-hot encoding
- **Numeric** → median imputation

In [ ]:
# ── Encode target ─────────────────────────────────────────────────────────────
label_order = ['Low', 'Medium', 'High']
le_target   = LabelEncoder()
le_target.fit(label_order)
y_raw = train_df['Target'].copy()
y     = le_target.transform(y_raw)

# ── Combine for consistent encoding ──────────────────────────────────────────
n_train  = len(train_df)
combined = pd.concat([train_df.drop(columns=['Target']), test_df],
                     axis=0, ignore_index=True)

# ── Ownership columns (ordinal) ───────────────────────────────────────────────
ownership_cols = [
    'motor_vehicle_insurance', 'has_mobile_money', 'has_credit_card',
    'has_loan_account', 'has_internet_banking', 'has_debit_card',
    'medical_insurance', 'funeral_insurance',
    'uses_friends_family_savings', 'uses_informal_lender'
]
ownership_map = {
    'Never had': 0,
    "Used to have but don't have now": 1,
    'Have now': 2
}
for col in ownership_cols:
    if col in combined.columns:
        combined[col] = combined[col].map(ownership_map).fillna(-1).astype(int)

# ── Yes/No columns (binary) ───────────────────────────────────────────────────
binary_cols = [
    'attitude_stable_business_environment', 'attitude_worried_shutdown',
    'compliance_income_tax', 'perception_insurance_doesnt_cover_losses',
    'perception_cannot_afford_insurance', 'current_problem_cash_flow',
    'offers_credit_to_customers', 'attitude_satisfied_with_achievement',
    'keeps_financial_records',
    'perception_insurance_companies_dont_insure_businesses_like_yours',
    'perception_insurance_important', 'has_insurance', 'covid_essential_service',
    'attitude_more_successful_next_year', 'problem_sourcing_money',
    'marketing_word_of_mouth', 'future_risk_theft_stock',
    'motivation_make_more_money', 'has_cellphone'
]
yes_no_map = {
    'Yes': 1, 'No': 0,
    'Yes, sometimes': 1, 'Yes, always': 1,
    "Don't know or N/A": -1,
    "Don?t know / doesn?t apply": -1,
    "Don't know": -1
}
for col in binary_cols:
    if col in combined.columns:
        combined[col] = combined[col].map(yes_no_map).fillna(-1)

# ── Gender ────────────────────────────────────────────────────────────────────
combined['owner_sex'] = combined['owner_sex'].map({'Male': 0, 'Female': 1}).fillna(-1)

# ── Country one-hot ───────────────────────────────────────────────────────────
combined = pd.get_dummies(combined, columns=['country'], drop_first=False)

# ── Numeric: median imputation ────────────────────────────────────────────────
for col in combined.select_dtypes(include=[np.number]).columns:
    median_val = combined[col].iloc[:n_train].median()
    combined[col] = combined[col].fillna(median_val)

# ── Remaining object columns ──────────────────────────────────────────────────
for col in combined.select_dtypes(include='object').columns:
    if col != 'ID':
        combined[col] = LabelEncoder().fit_transform(combined[col].astype(str))

print(f"✅ Preprocessing done — shape: {combined.shape}")

## 4. ⚙️ Feature Engineering

We create domain-aware features and numerical transformations:
- **Composite scores**: financial access, informal finance, attitude
- **Log transforms**: reduce skewness in monetary columns
- **Interaction features**: multiplicative combinations of top features
- **Ratios**: efficiency and profitability proxies

In [ ]:
# ── Financial access score ────────────────────────────────────────────────────
fin_cols = ['has_mobile_money', 'has_credit_card', 'has_loan_account',
            'has_internet_banking', 'has_debit_card', 'has_insurance',
            'motor_vehicle_insurance', 'medical_insurance', 'funeral_insurance']
combined['fin_access_score'] = combined[[c for c in fin_cols if c in combined.columns]].clip(lower=0).sum(axis=1)

# ── Informal finance usage ────────────────────────────────────────────────────
inf_cols = ['uses_friends_family_savings', 'uses_informal_lender']
combined['informal_finance_score'] = combined[[c for c in inf_cols if c in combined.columns]].clip(lower=0).sum(axis=1)

# ── Positive attitude score ───────────────────────────────────────────────────
att_cols = ['attitude_stable_business_environment', 'attitude_satisfied_with_achievement',
            'attitude_more_successful_next_year', 'motivation_make_more_money']
combined['positive_attitude_score'] = combined[[c for c in att_cols if c in combined.columns]].clip(lower=0).sum(axis=1)

# ── Business maturity ─────────────────────────────────────────────────────────
if 'business_age_years' in combined.columns and 'business_age_months' in combined.columns:
    combined['business_age_total_months'] = (
        combined['business_age_years'].fillna(0) * 12 +
        combined['business_age_months'].fillna(0)
    )

# ── Log transforms on monetary columns ───────────────────────────────────────
for col in ['personal_income', 'business_expenses', 'business_turnover']:
    if col in combined.columns:
        combined[f'log_{col}'] = np.log1p(combined[col].clip(lower=0))

# ── Ratios ────────────────────────────────────────────────────────────────────
if 'personal_income' in combined.columns and 'business_expenses' in combined.columns:
    combined['income_expense_ratio'] = (
        combined['personal_income'] / (combined['business_expenses'] + 1)
    ).clip(upper=100)

if 'business_turnover' in combined.columns and 'business_expenses' in combined.columns:
    combined['turnover_expense_ratio'] = (
        combined['business_turnover'] / (combined['business_expenses'] + 1)
    ).clip(upper=100)

if 'business_turnover' in combined.columns and 'personal_income' in combined.columns:
    combined['turnover_income_ratio'] = (
        combined['business_turnover'] / (combined['personal_income'] + 1)
    ).clip(upper=100)

# ── Interaction features ──────────────────────────────────────────────────────
combined['fin_x_attitude']    = combined['fin_access_score'] * combined['positive_attitude_score']
combined['fin_x_records']     = combined['fin_access_score'] * combined['keeps_financial_records'].clip(lower=0)
combined['attitude_x_age']    = combined['positive_attitude_score'] * combined['owner_age'].fillna(0)

# ── Insurance awareness score ─────────────────────────────────────────────────
ins_cols = ['perception_insurance_important', 'perception_cannot_afford_insurance',
            'perception_insurance_doesnt_cover_losses']
combined['insurance_awareness'] = combined[[c for c in ins_cols if c in combined.columns]].clip(lower=0).sum(axis=1)

print(f"✅ Feature engineering done — shape: {combined.shape}")

## 5. ✂️ Split Back into Train / Test

In [ ]:
feature_cols = [c for c in combined.columns if c != 'ID']

X_train = combined.iloc[:n_train][feature_cols].values.astype(np.float32)
X_test  = combined.iloc[n_train:][feature_cols].values.astype(np.float32)

# Compute class weights to handle imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y)

SKF = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {le_target.classes_}")

## 6. 🔎 Optuna Hyperparameter Tuning

We use **Optuna** (Bayesian optimization) to find the best parameters for each model.
- `n_trials=50` per model is a good balance between speed and quality
- Increase to 100+ for better results if time allows
- The objective function maximizes the weighted F1 score on cross-validation

In [ ]:
N_TRIALS = 50  # Increase to 100 for better tuning (takes more time)

# ── Tune XGBoost ──────────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = dict(
        n_estimators       = trial.suggest_int('n_estimators', 300, 1200),
        learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        max_depth          = trial.suggest_int('max_depth', 4, 10),
        subsample          = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree   = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_weight   = trial.suggest_int('min_child_weight', 1, 10),
        reg_alpha          = trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        reg_lambda         = trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        gamma              = trial.suggest_float('gamma', 0, 1),
        use_label_encoder  = False,
        eval_metric        = 'mlogloss',
        random_state       = SEED,
        n_jobs             = -1
    )
    model  = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y, cv=SKF,
                             scoring='f1_weighted', fit_params={'sample_weight': sample_weights})
    return scores.mean()

print("🔎 Tuning XGBoost...")
study_xgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"✅ XGBoost best F1: {study_xgb.best_value:.4f}")
print(f"   Best params: {best_xgb_params}")

In [ ]:
# ── Tune LightGBM ─────────────────────────────────────────────────────────────
def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 300, 1200),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 20, 150),
        max_depth         = trial.suggest_int('max_depth', 4, 12),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        random_state      = SEED,
        n_jobs            = -1,
        verbose           = -1
    )
    model  = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train, y, cv=SKF,
                             scoring='f1_weighted', fit_params={'sample_weight': sample_weights})
    return scores.mean()

print("🔎 Tuning LightGBM...")
study_lgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"✅ LightGBM best F1: {study_lgb.best_value:.4f}")
print(f"   Best params: {best_lgb_params}")

In [ ]:
# ── Tune CatBoost ─────────────────────────────────────────────────────────────
def cat_objective(trial):
    params = dict(
        iterations        = trial.suggest_int('iterations', 300, 1200),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        depth             = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg       = trial.suggest_float('l2_leaf_reg', 1e-2, 10, log=True),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        random_seed       = SEED,
        loss_function     = 'MultiClass',
        verbose           = False
    )
    model  = CatBoostClassifier(**params)
    scores = cross_val_score(model, X_train, y, cv=SKF,
                             scoring='f1_weighted', fit_params={'sample_weight': sample_weights})
    return scores.mean()

print("🔎 Tuning CatBoost...")
study_cat = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_cat.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"✅ CatBoost best F1: {study_cat.best_value:.4f}")
print(f"   Best params: {best_cat_params}")

## 7. 🧠 Train Tuned Models with OOF Predictions

We retrain each model with the best hyperparameters found by Optuna.
Out-of-Fold (OOF) predictions are collected for stacking.
Class weights are applied to handle imbalance.

In [ ]:
N_CLASSES = 3

oof_xgb  = np.zeros((n_train, N_CLASSES))
oof_lgb  = np.zeros((n_train, N_CLASSES))
oof_cat  = np.zeros((n_train, N_CLASSES))

test_xgb = np.zeros((len(X_test), N_CLASSES))
test_lgb = np.zeros((len(X_test), N_CLASSES))
test_cat = np.zeros((len(X_test), N_CLASSES))

# ── XGBoost ───────────────────────────────────────────────────────────────────
print("Training XGBoost with best params...")
for fold, (trn_idx, val_idx) in enumerate(SKF.split(X_train, y)):
    sw_fold = sample_weights[trn_idx]
    m = xgb.XGBClassifier(
        **best_xgb_params,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=SEED, n_jobs=-1
    )
    m.fit(X_train[trn_idx], y[trn_idx],
          sample_weight=sw_fold,
          eval_set=[(X_train[val_idx], y[val_idx])],
          early_stopping_rounds=50, verbose=False)
    oof_xgb[val_idx]  = m.predict_proba(X_train[val_idx])
    test_xgb         += m.predict_proba(X_test) / N_FOLDS
    f1 = f1_score(y[val_idx], oof_xgb[val_idx].argmax(1), average='weighted')
    print(f"  Fold {fold+1} XGB F1: {f1:.4f}")

xgb_f1 = f1_score(y, oof_xgb.argmax(1), average='weighted')
print(f"  ➜ XGB OOF F1: {xgb_f1:.4f}\n")

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
print("Training LightGBM with best params...")
for fold, (trn_idx, val_idx) in enumerate(SKF.split(X_train, y)):
    sw_fold = sample_weights[trn_idx]
    m = lgb.LGBMClassifier(
        **best_lgb_params,
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    m.fit(X_train[trn_idx], y[trn_idx],
          sample_weight=sw_fold,
          eval_set=[(X_train[val_idx], y[val_idx])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx]  = m.predict_proba(X_train[val_idx])
    test_lgb         += m.predict_proba(X_test) / N_FOLDS
    f1 = f1_score(y[val_idx], oof_lgb[val_idx].argmax(1), average='weighted')
    print(f"  Fold {fold+1} LGB F1: {f1:.4f}")

lgb_f1 = f1_score(y, oof_lgb.argmax(1), average='weighted')
print(f"  ➜ LGB OOF F1: {lgb_f1:.4f}\n")

In [ ]:
# ── CatBoost ──────────────────────────────────────────────────────────────────
print("Training CatBoost with best params...")
for fold, (trn_idx, val_idx) in enumerate(SKF.split(X_train, y)):
    sw_fold = sample_weights[trn_idx]
    m = CatBoostClassifier(
        **best_cat_params,
        loss_function='MultiClass',
        random_seed=SEED, verbose=False,
        early_stopping_rounds=50
    )
    m.fit(X_train[trn_idx], y[trn_idx],
          sample_weight=sw_fold,
          eval_set=(X_train[val_idx], y[val_idx]),
          verbose=False)
    oof_cat[val_idx]  = m.predict_proba(X_train[val_idx])
    test_cat         += m.predict_proba(X_test) / N_FOLDS
    f1 = f1_score(y[val_idx], oof_cat[val_idx].argmax(1), average='weighted')
    print(f"  Fold {fold+1} CAT F1: {f1:.4f}")

cat_f1 = f1_score(y, oof_cat.argmax(1), average='weighted')
print(f"  ➜ CAT OOF F1: {cat_f1:.4f}\n")

## 8. 🧱 Stacking Meta-Model

We use the OOF probability predictions from all 3 models as input features
to train a **Logistic Regression meta-model**.

This learns **how to best combine** the base models for maximum F1.

In [ ]:
# Stack OOF probabilities as meta-features (9 features total: 3 models × 3 classes)
meta_train = np.hstack([oof_xgb, oof_lgb, oof_cat])
meta_test  = np.hstack([test_xgb, test_lgb, test_cat])

# Train meta-model with cross-validation to avoid overfitting
meta_model = LogisticRegression(
    max_iter=2000,
    C=1.0,
    class_weight='balanced',
    random_state=SEED,
    solver='lbfgs',
    multi_class='multinomial'
)
meta_model.fit(meta_train, y)

# Evaluate stacking
stack_preds = meta_model.predict(meta_train)
stack_f1    = f1_score(y, stack_preds, average='weighted')

print(f"✅ Stacking OOF F1: {stack_f1:.4f}")
print()
print("Classification Report:")
print(classification_report(y, stack_preds, target_names=le_target.classes_))

## 9. 🔄 Pseudo-Labeling

High-confidence test predictions (probability > 0.90) are added to the training set.
This gives the models more signal, especially for underrepresented classes.

We then **retrain one final LightGBM** on this augmented dataset.

In [ ]:
# Get test probabilities from stacking meta-model
test_proba_stack = meta_model.predict_proba(meta_test)
max_proba        = test_proba_stack.max(axis=1)

CONFIDENCE_THRESHOLD = 0.90
confident_mask = max_proba >= CONFIDENCE_THRESHOLD
print(f"High-confidence samples (>{CONFIDENCE_THRESHOLD}): {confident_mask.sum()} / {len(X_test)}")

if confident_mask.sum() > 0:
    X_pseudo = X_test[confident_mask]
    y_pseudo = test_proba_stack[confident_mask].argmax(axis=1)

    # Augment training data
    X_aug = np.vstack([X_train, X_pseudo])
    y_aug = np.concatenate([y, y_pseudo])
    sw_aug = compute_sample_weight(class_weight='balanced', y=y_aug)

    print(f"Augmented train size: {len(X_aug)} (was {len(X_train)})")

    # Retrain LightGBM on augmented data (fastest model for this step)
    print("\nRetraining LightGBM on augmented data...")
    pseudo_lgb_params = {**best_lgb_params, 'random_state': SEED, 'n_jobs': -1, 'verbose': -1}
    model_pseudo = lgb.LGBMClassifier(**pseudo_lgb_params)
    model_pseudo.fit(X_aug, y_aug, sample_weight=sw_aug)

    test_pseudo = model_pseudo.predict_proba(X_test)
    print("✅ Pseudo-label model trained.")
else:
    test_pseudo = test_lgb  # fallback
    print("No confident samples found — using standard LGB predictions.")

## 10. 🏆 Final Ensemble

We combine all predictions:
- Stacking meta-model probabilities (main signal)
- Pseudo-label LightGBM probabilities (extra signal)

The stacking model carries more weight (0.7) since it was validated on OOF.

In [ ]:
# Weighted blend: stacking (70%) + pseudo-lgb (30%)
final_proba = 0.7 * test_proba_stack + 0.3 * test_pseudo
final_preds = final_proba.argmax(axis=1)
final_labels = le_target.inverse_transform(final_preds)

print("Final prediction distribution:")
unique, counts = np.unique(final_labels, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {cls}: {cnt} ({cnt/len(final_labels)*100:.1f}%)")

## 11. 📤 Generate Submission

In [ ]:
submission = pd.DataFrame({
    'ID':     test_df['ID'].values,
    'Target': final_labels
})

submission.to_csv('submission_advanced.csv', index=False)
print(f"✅ Submission saved: submission_advanced.csv")
print(f"   Shape: {submission.shape}")
submission.head(10)

## 12. 📊 Final Performance Summary

In [ ]:
summary = pd.DataFrame({
    'Model':   ['XGBoost (tuned)', 'LightGBM (tuned)', 'CatBoost (tuned)', '⭐ Stacking', '⭐⭐ Final Blend'],
    'OOF F1':  [xgb_f1, lgb_f1, cat_f1, stack_f1, None],
    'Note':    ['base model', 'base model', 'base model',
                'LR on OOF proba', 'Stacking + Pseudo-LGB']
})
print(summary.to_string(index=False))
print("\n✅ submission_advanced.csv is ready to upload!")

---
## 💡 Additional Ideas to Push Further

- **Increase `N_TRIALS`** to 100–200 for better Optuna tuning
- **Feature selection**: Remove features with near-zero importance using `model.feature_importances_`
- **Target encoding**: Replace `country` one-hot with mean target encoding per country per class
- **RandomForest** as 4th base model in the stacking ensemble
- **Lower confidence threshold** (e.g., 0.85) to add more pseudo-labeled samples
- **Multiple seeds**: Average predictions across different random seeds for stability